# kumo-tabular on a local NIM

One table. You have rows whose answer is known and rows whose answer you want,
and both go in one request.

This needs only the base install, `pip install structured-data-models-client`,
and a `kumo-tabular` NIM you point it at. The relational model is a separate
notebook, [`relational.ipynb`](relational.ipynb).

In [1]:
import os

import pandas as pd
from sdm_client import KumoTabular, Request, SDMClient

# Point this at your own deployment, or set the variable instead.
TABULAR_URL = os.environ.get('SDM_TABULAR_URL', 'http://localhost:8000')

## One table

You have rows where the answer is known, and rows you want scored. The rows
being scored need not appear in the first half, which is the point of the
model.


In [2]:
labelled = pd.DataFrame(
    {
        'age': [31.0, 44.5, 28.0, 51.0, 39.0, 47.0, 25.0, 58.0],
        'spend': [120.0, 430.0, 90.0, 510.0, 260.0, 480.0, 70.0, 620.0],
        'churn': [0, 1, 0, 1, 0, 1, 0, 1],
    }
)
to_score = pd.DataFrame({'age': [33.0, 47.0], 'spend': [150.0, 480.0]})

request = Request(
    context=labelled,
    predict=to_score,
    target='churn',
    task='classification',
)
request

Request(classification, target='churn', 8 context rows, 2 to score, single-table)

In [3]:
with SDMClient(url=TABULAR_URL) as client:
    print('the endpoint serves:', client.models())
    result = client.predict(request, model=KumoTabular())

result.to_pandas()

the endpoint serves: ['kumo-tabular']


,row_index,prediction,probability_0,probability_1
0,0,0,0.980807,0.019192
1,1,1,0.003052,0.996948


Join on the key the result carries, or on `row_index`. **Never on row
order**. The endpoint is not required to answer in the order you asked, and a
request split across several calls certainly will not.


In [4]:
answers = result.to_pandas()
to_score.reset_index(names='row_index').merge(answers, on='row_index')

,row_index,age,spend,prediction,probability_0,probability_1
0,0,33.0,150.0,0,0.980807,0.019192
1,1,47.0,480.0,1,0.003052,0.996948


## The same question, asked in PQL

`Request` names the two halves directly. The Predictive Query Language says
what to predict and for which rows instead, and on one table it resolves to the
same kind of request: the rows named by `indices` are scored, and the rest of
the table is the context.

Use it when the question is easier to write down than to split by hand, or when
the same text has to work for both models.

In [5]:
from sdm_client import PredictiveQuery, Schema

customers = pd.DataFrame(
    {
        'customer_id': range(40),
        'age': [25.0 + i % 40 for i in range(40)],
        'spend': [100.0 + 9 * i for i in range(40)],
        'churn': [i % 2 for i in range(40)],
    }
)

schema = Schema.from_data(
    {'customers': customers}, primary_keys={'customers': 'customer_id'}
)
query = PredictiveQuery(
    'PREDICT customers.churn FOR EACH customers.customer_id', schema
)
request = query.materialize(indices=[3, 7, 11], model=KumoTabular())

with SDMClient(url=TABULAR_URL) as client:
    print(client.predict(request, model=KumoTabular()).to_pandas())

   row_index  id  prediction  probability_0  probability_1
0          0   3           0       0.578590       0.421410
1          1   7           0       0.574676       0.425324
2          2  11           0       0.566255       0.433745


/tmp/ipykernel_50300/2821833205.py:18: SDMWarning: customers.churn holds two whole numbers, which is how a binary label is usually written, so this is being asked as a classification. If they are quantities rather than labels, pass task="regression" to ask for one; pass task="classification" to say this was meant and silence this.
  request = query.materialize(indices=[3, 7, 11], model=KumoTabular())


`indices` is required: on one table a query can only name rows that are already
there, so something has to say which of them are the question rather than the
context.

Almost every clause the relational language has is refused here: no
aggregations, no time windows, no `RANK`, no `ASSUMING`. One table has nothing
to aggregate over and no history to look back through, and each refusal says
so.

It also warns. `Request` makes you state the task; a query infers it, and a
column of two whole numbers could be labels or quantities. The warning says
which reading it took and how to say the other: pass
`materialize(task='classification')` to confirm it and silence this, or
`task='regression'` to overrule it.
